# Projeto REtinopatia

## Configurações Iniciais, Importações



In [1]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

# TensorFlow e Keras
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Scikit-learn para divisão inteligente e pesos
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils import class_weight

# Configurando as "Seeds" (Sementes) para garantir a reprodutibilidade da pesquisa
SEED = 42

# Mantém o valor da semente registrado no ambiente
os.environ["PYTHONHASHSEED"] = str(SEED)

# Configura de uma vez as sementes do Python, NumPy e TensorFlow
tf.keras.utils.set_random_seed(SEED)

# Tenta utilizar operações determinísticas no TensorFlow
try:
    tf.config.experimental.enable_op_determinism()

    print(
        "Operações determinísticas ativadas."
    )

except Exception as erro:
    print(
        "Não foi possível ativar determinismo completo:",
        erro
    )

print(
    "Bibliotecas importadas e sementes "
    "configuradas com sucesso!"
)

Operações determinísticas ativadas.
Bibliotecas importadas e sementes configuradas com sucesso!


## Conexão com o Drive e Preparação do Ambiente

In [ ]:
from google.colab import drive
import os

# 1. Conecta ao Google Drive
drive.mount('/content/drive')

# 2. Definição dos caminhos
caminho_zip_principal = '/content/drive/MyDrive/Datasets/diabetic-retinopathy-detection.zip'
pasta_base = '/content/dataset_retinopatia'
pasta_imagens = os.path.join(pasta_base, 'train')

print(" Verificando o ambiente de dados...")

# 3. Extração Inteligente (Pula a etapa se as imagens já estiverem lá)
if not os.path.exists(pasta_imagens):
    os.makedirs(pasta_base, exist_ok=True)

    print("Extraindo ZIP principal...")
    !unzip -q -o "{caminho_zip_principal}" -d "{pasta_base}"

    print("Extraindo CSV de labels...")
    !unzip -q -o "{pasta_base}/trainLabels.csv.zip" -d "{pasta_base}"

    print("Juntando e extraindo as imagens de treino (Isso vai levar alguns minutos)...")
    !cat {pasta_base}/train.zip.* > {pasta_base}/train_completo.zip
    !unzip -q -o {pasta_base}/train_completo.zip -d "{pasta_base}"

    print(" Extração concluída!")
else:
    print(" Os dados já foram extraídos anteriormente. Pulando esta etapa para economizar tempo.")

Mounted at /content/drive
 Verificando o ambiente de dados...
Extraindo ZIP principal...


## Carregamento do CSV e Divisão Estratificada


In [ ]:
caminho_csv = os.path.join(
    pasta_base,
    "trainLabels.csv"
)

# 1. LEITURA DO CSV

dados = pd.read_csv(caminho_csv)

# Adiciona .jpeg aos nomes das imagens
dados["image"] = (
    dados["image"]
    .astype(str)
    .apply(lambda nome: f"{nome}.jpeg")
)

# 1.1 VERIFICA O PADRÃO DOS NOMES DAS IMAGENS

# Formato esperado:
# 123_left.jpeg
# 123_right.jpeg
padrao_valido = dados["image"].str.match(
    r"^.+_(?:left|right)\.jpeg$"
)

if not padrao_valido.all():
    imagens_fora_do_padrao = dados.loc[
        ~padrao_valido,
        "image"
    ]

    print(
        "\nExemplos de imagens fora do padrão esperado:"
    )

    print(
        imagens_fora_do_padrao
        .head(20)
        .to_string(index=False)
    )

    raise ValueError(
        f"{len(imagens_fora_do_padrao)} imagens "
        "não seguem o padrão esperado de nomes."
    )

print(
    "\nTodos os nomes seguem o padrão "
    "'identificador_left.jpeg' ou "
    "'identificador_right.jpeg'."
)

# 2. PRESERVA O GRAU ORIGINAL

# Converte os níveis para números inteiros.
# errors="raise" faz o código parar se houver algum valor inválido.
dados["level_original"] = pd.to_numeric(
    dados["level"],
    errors="raise"
).astype("int8")

# Verifica se existem somente os graus esperados
niveis_permitidos = {0, 1, 2, 3, 4}

niveis_encontrados = set(
    dados["level_original"].unique()
)

niveis_invalidos = (
    niveis_encontrados - niveis_permitidos
)

if niveis_invalidos:
    raise ValueError(
        f"Níveis inválidos encontrados: {niveis_invalidos}"
    )

# 3. CRIA A CLASSE BINÁRIA USADA PELO MODELO

# Grau 0 vira classe 0: saudável
# Graus 1, 2, 3 e 4 viram classe 1: com retinopatia
dados["target"] = (
    dados["level_original"] > 0
).astype("int8")

# 4. EXTRAI O IDENTIFICADOR DO PACIENTE

# Exemplo:
# 123_left.jpeg  -> paciente 123
# 123_right.jpeg -> paciente 123
dados["patient_id"] = (
    dados["image"]
    .str.replace(
        r"\.jpeg$",
        "",
        regex=True
    )
    .str.replace(
        r"_(left|right)$",
        "",
        regex=True
    )
)

print("\nExemplos de imagens, pacientes e classificações:")

print(
    dados[
        [
            "image",
            "patient_id",
            "level_original",
            "target"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# 5. DIVISÃO AGRUPADA POR PACIENTE

divisor = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

indices_treino, indices_val = next(
    divisor.split(
        X=dados["image"],

        # Estratifica usando os cinco graus originais
        y=dados["level_original"],

        # Mantém os dois olhos do mesmo paciente juntos
        groups=dados["patient_id"]
    )
)

df_treino = dados.iloc[
    indices_treino
].copy()

df_val = dados.iloc[
    indices_val
].copy()

# 6. VERIFICA SE HÁ PACIENTES REPETIDOS

pacientes_treino = set(
    df_treino["patient_id"]
)

pacientes_val = set(
    df_val["patient_id"]
)

assert pacientes_treino.isdisjoint(
    pacientes_val
), (
    "Erro: existem pacientes repetidos "
    "entre treino e validação."
)

print(
    "\nNenhum paciente aparece simultaneamente "
    "no treino e na validação."
)

# 7. CONVERTE APENAS O TARGET PARA TEXTO

# O flow_from_dataframe com class_mode="binary"
# trabalha corretamente com as classes como strings.
df_treino["target"] = (
    df_treino["target"]
    .astype(str)
)

df_val["target"] = (
    df_val["target"]
    .astype(str)
)

# 7.1 SALVA OS CONJUNTOS NO GOOGLE DRIVE

pasta_splits = (
    "/content/drive/MyDrive/"
    "ProjetoRetinopatia/splits"
)

# Cria a pasta caso ainda não exista
os.makedirs(
    pasta_splits,
    exist_ok=True
)

caminho_split_treino = os.path.join(
    pasta_splits,
    f"treino_seed_{SEED}.csv"
)

caminho_split_validacao = os.path.join(
    pasta_splits,
    f"validacao_seed_{SEED}.csv"
)

df_treino.to_csv(
    caminho_split_treino,
    index=False
)

df_val.to_csv(
    caminho_split_validacao,
    index=False
)

print("\nSplits salvos com sucesso:")

print(
    "Treino:",
    caminho_split_treino
)

print(
    "Validação:",
    caminho_split_validacao
)

# 8. RESUMO DOS CONJUNTOS

print(
    f"\nTotal de imagens originais: "
    f"{len(dados)}"
)

print(
    f"Imagens separadas para TREINO: "
    f"{len(df_treino)}"
)

print(
    f"Imagens separadas para VALIDAÇÃO: "
    f"{len(df_val)}"
)

# 8.1 DISTRIBUIÇÃO DAS CLASSES NOS CONJUNTOS

print("\nDistribuição dos graus originais no TREINO:")

print(
    df_treino["level_original"]
    .value_counts(normalize=True)
    .sort_index()
    .apply(
        lambda valor: f"{valor * 100:.2f}%"
    )
)


print("\nDistribuição dos graus originais na VALIDAÇÃO:")

print(
    df_val["level_original"]
    .value_counts(normalize=True)
    .sort_index()
    .apply(
        lambda valor: f"{valor * 100:.2f}%"
    )
)


print("\nDistribuição binária no TREINO:")

print(
    df_treino["target"]
    .value_counts(normalize=True)
    .sort_index()
    .apply(
        lambda valor: f"{valor * 100:.2f}%"
    )
)


print("\nDistribuição binária na VALIDAÇÃO:")

print(
    df_val["target"]
    .value_counts(normalize=True)
    .sort_index()
    .apply(
        lambda valor: f"{valor * 100:.2f}%"
    )
)

## Inspeção visual

In [ ]:
amostra = dados.sample(
    n=min(20, len(dados)),
    random_state=SEED
)

for nome in amostra["image"]:
    caminho = os.path.join(pasta_imagens, nome)
    img = cv2.imread(caminho)

    if img is None:
        print(f"Erro ao ler: {nome}")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_blur = cv2.GaussianBlur(gray, (5, 5), 0)

    _, thresh = cv2.threshold(
        gray_blur,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    pontos = cv2.findNonZero(thresh)

    if pontos is None:
      print(f"Máscara vazia para a imagem: {nome}")
      recorte = img
    else:
      x, y, w, h = cv2.boundingRect(pontos)
      recorte = img[y:y+h, x:x+w]

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(thresh, cmap="gray")
    plt.title("Máscara de Otsu")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(recorte, cv2.COLOR_BGR2RGB))
    plt.title("Recorte")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

## Pré-processamento Offline (Evitar 40 min por época)

In [ ]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import os

# Pasta desta versão do pré-processamento
pasta_imagens_otimizadas = "/content/dataset_otimizado_otsu_v1"

os.makedirs(
    pasta_imagens_otimizadas,
    exist_ok=True
)


def processar_e_salvar(img_nome):
    """
    Lê, recorta, redimensiona e salva uma imagem.

    Retorna:
        (nome_da_imagem, status_do_processamento)
    """

    caminho_in = os.path.join(
        pasta_imagens,
        img_nome
    )

    caminho_out = os.path.join(
        pasta_imagens_otimizadas,
        img_nome
    )

    # Se a imagem já foi processada, não processa novamente
    if os.path.exists(caminho_out):
        return img_nome, "existente"

    # Tenta ler a imagem
    img = cv2.imread(caminho_in)

    if img is None:
        return img_nome, "erro_leitura"

    # Converte para escala de cinza somente para criar a máscara
    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    # Suaviza pequenos ruídos da máscara.
    # A imagem original não é desfocada.
    gray_blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # Cria a máscara usando o método de Otsu
    _, thresh = cv2.threshold(
        gray_blur,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Localiza os pixels pertencentes à região detectada
    pontos = cv2.findNonZero(thresh)

    # Se nenhum ponto for encontrado, registra a falha
    if pontos is None:
        return img_nome, "mascara_vazia"

    # Calcula a caixa delimitadora da retina
    x, y, w, h = cv2.boundingRect(pontos)

    if w <= 0 or h <= 0:
        return img_nome, "recorte_invalido"

    # Recorta a imagem original
    recortada = img[
        y:y + h,
        x:x + w
    ]

    if recortada.size == 0:
        return img_nome, "recorte_invalido"

    # Redimensiona para o tamanho esperado pela EfficientNetB0
    img_final = cv2.resize(
        recortada,
        (224, 224),
        interpolation=cv2.INTER_AREA
    )

    # Salva a imagem e guarda o resultado da operação
    sucesso = cv2.imwrite(
        caminho_out,
        img_final
    )

    if not sucesso:
        return img_nome, "erro_escrita"

    return img_nome, "processada"


print("Iniciando recorte otimizado em paralelo...")

# Limita o número de threads para evitar sobrecarga
numero_workers = min(
    8,
    os.cpu_count() or 1
)

with ThreadPoolExecutor(
    max_workers=numero_workers
) as executor:

    resultados = list(
        tqdm(
            executor.map(
                processar_e_salvar,
                dados["image"].values
            ),
            total=len(dados)
        )
    )

print("\nProcessamento concluído!")


# Organiza os resultados em uma tabela
resultados_df = pd.DataFrame(
    resultados,
    columns=["image", "status"]
)

print("\nResumo do processamento:")
print(
    resultados_df["status"]
    .value_counts()
)


# Define quais estados representam falhas
status_de_erro = [
    "erro_leitura",
    "mascara_vazia",
    "recorte_invalido",
    "erro_escrita"
]

falhas = resultados_df[
    resultados_df["status"].isin(status_de_erro)
]


# Interrompe o notebook caso alguma imagem tenha falhado
if not falhas.empty:
    print("\nExemplos de imagens que apresentaram problemas:")
    print(
        falhas.head(20).to_string(index=False)
    )

    raise RuntimeError(
        f"{len(falhas)} imagens falharam no pré-processamento."
    )


# Confirma se todas as imagens esperadas existem na pasta final
imagens_ausentes = [
    nome
    for nome in dados["image"].values
    if not os.path.isfile(
        os.path.join(
            pasta_imagens_otimizadas,
            nome
        )
    )
]

if imagens_ausentes:
    print("\nExemplos de imagens ausentes:")
    print(imagens_ausentes[:20])

    raise RuntimeError(
        f"{len(imagens_ausentes)} imagens não foram encontradas "
        "na pasta processada."
    )

print(
    "\nTodas as imagens foram processadas "
    "ou já existiam corretamente!"
)

## Data Generators (Pipelines de Dados e Data Augmentation)

In [ ]:
# Gerador de TREINO (Com Data Augmentation)
gerador_treino = ImageDataGenerator(
    rotation_range=20,          # Roda a imagem até 20 graus
    horizontal_flip=True,       # Espelhamento horizontal
    vertical_flip=True,         # Espelhamento vertical
    zoom_range=0.1,             # Aplica um leve zoom aleatório
    fill_mode='constant',       # Preenche as margens com a cor preta após a rotação
    cval=0
)

# Gerador de VALIDAÇÃO (Limpo, sem Data Augmentation e SEM rescale para a EfficientNet)
gerador_val = ImageDataGenerator()

# Parâmetros gerais
# Como temos muita memória de GPU disponível, vamos aumentar o lote drasticamente
# para acelerar o treino. Se der erro de falta de memória (OOM), diminua para 64 ou 32.
tamanho_lote = 128

print("\nA preparar imagens de TREINO...")
treino_data = gerador_treino.flow_from_dataframe(
    dataframe=df_treino,
    directory=pasta_imagens_otimizadas,
    x_col='image',
    y_col='target',
    class_mode="binary",
    target_size=(224, 224),
    batch_size=tamanho_lote,
    seed=SEED,
    shuffle=True # Fundamental embaralhar no treino
)

print("\nA preparar imagens de VALIDAÇÃO...")
val_data = gerador_val.flow_from_dataframe(
    dataframe=df_val,
    directory=pasta_imagens_otimizadas,
    x_col='image',
    y_col='target',
    class_mode="binary",
    target_size=(224, 224),
    batch_size=tamanho_lote,
    seed=SEED,
    shuffle=False # Nunca embaralhar a validação para podermos criar a matriz de confusão correta depois
)

# Verificação da escala das imagens
x_treino, y_treino = next(treino_data)
x_val, y_val = next(val_data)

print("\nVerificação da escala das imagens:")

print(
    "Treino:",
    "tipo =", x_treino.dtype,
    "| mínimo =", x_treino.min(),
    "| máximo =", x_treino.max()
)

print(
    "Validação:",
    "tipo =", x_val.dtype,
    "| mínimo =", x_val.min(),
    "| máximo =", x_val.max()
)

# A EfficientNet do Keras espera imagens aproximadamente entre 0 e 255
assert x_treino.max() > 1.0, (
    "Erro: as imagens de treino parecem estar normalizadas entre 0 e 1."
)

assert x_val.max() > 1.0, (
    "Erro: as imagens de validação parecem estar normalizadas entre 0 e 1."
)

# Retorna os geradores ao início antes do treinamento
treino_data.reset()
val_data.reset()

print("Escala das imagens verificada com sucesso!")


## Cálculo dos Pesos das Classes

In [ ]:
print("A calcular os pesos matemáticos das classes...")

# Converte a coluna target para números inteiros
classes_reais_treino = (
    df_treino["target"]
    .astype(int)
    .values
)

# Descobre quais classes realmente existem no treino
# Neste projeto, o esperado é: [0, 1]
classes = np.unique(classes_reais_treino)

# Verifica se as duas classes estão presentes
if set(classes) != {0, 1}:
    raise ValueError(
        f"Classes inesperadas no treino: {classes}. "
        "O esperado era encontrar as classes 0 e 1."
    )

# Calcula os pesos de acordo com a quantidade
# de exemplos de cada classe
pesos = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=classes_reais_treino
)

# Associa cada classe diretamente ao seu peso
# Exemplo: {0: 0.65, 1: 2.10}
pesos_dicionario = dict(
    zip(
        classes.tolist(),
        pesos.tolist()
    )
)

# Nomes corretos das classes binárias
nomes_classes = {
    0: "Sem retinopatia",
    1: "Com retinopatia, graus originais 1 a 4"
}

print("\nPesos calculados:")

for classe, peso in pesos_dicionario.items():
    print(
        f"Classe {classe} "
        f"({nomes_classes[classe]}): "
        f"peso {peso:.4f}"
    )

print(
    "\nPesos calculados com sucesso! "
    "A classe menos frequente receberá maior peso durante o treinamento."
)

## Arquitetura do Modelo (Transfer Learning com EfficientNetB0)

In [ ]:
print("A carregar a EfficientNetB0 e a construir o modelo...")


def criar_metricas():
    """
    Retorna novas instâncias das métricas para cada compilação.
    """
    return [
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="sensitivity"
        ),
        tf.keras.metrics.AUC(
            name="roc_auc",
            curve="ROC"
        ),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]


# Carrega a EfficientNetB0 sem a camada classificadora original
modelo_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Fase 1: congela a EfficientNet e treina apenas a cabeça
modelo_base.trainable = False

modelo = models.Sequential([
    modelo_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(
        256,
        activation="relu"
    ),
    layers.Dropout(0.5),
    layers.Dense(
        1,
        activation="sigmoid"
    )
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=criar_metricas()
)

modelo.summary()

print("Arquitetura montada com sucesso!")

## Callbacks (Monitorização do Treino)

In [ ]:
checkpoint = ModelCheckpoint(
    "melhor_modelo_retinopatia.weights.h5",
    monitor="val_pr_auc",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_pr_auc",
    mode="max",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

meus_callbacks = [
    checkpoint,
    early_stop,
    reduce_lr
]

## Treino do Modelo (Fase 1 - Apenas a cabeça da rede)

In [ ]:
print(" Ininiando o treinamento...")

historico = modelo.fit(
    treino_data,
    validation_data=val_data,
    epochs=15,
    callbacks=meus_callbacks,
    class_weight=pesos_dicionario # O superpoder para lidar com o desbalanceamento
)

print("\n Treino da Fase 1 concluído com sucesso!")

print("Carregando o melhor checkpoint da Fase 1 para avaliação...")

modelo.load_weights(
    "melhor_modelo_retinopatia.weights.h5"
)

## Avaliação, Gráficos e Matriz de Confusão

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve
)

import seaborn as sns
import numpy as np

# ============================================================
# 1. GRÁFICOS DO TREINAMENTO
# ============================================================

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)

plt.plot(
    historico.history["accuracy"],
    label="Treino",
    linewidth=2
)

plt.plot(
    historico.history["val_accuracy"],
    label="Validação",
    linewidth=2
)

plt.title("Evolução da Acurácia")
plt.xlabel("Épocas")
plt.ylabel("Acurácia")
plt.legend()


plt.subplot(1, 2, 2)

plt.plot(
    historico.history["loss"],
    label="Treino",
    linewidth=2
)

plt.plot(
    historico.history["val_loss"],
    label="Validação",
    linewidth=2
)

plt.title("Evolução do Erro")
plt.xlabel("Épocas")
plt.ylabel("Erro")
plt.legend()

plt.tight_layout()
plt.show()


# ============================================================
# 2. ESCOLHA DO LIMIAR NA VALIDAÇÃO
# ============================================================

print("\nGerando probabilidades na validação...")

val_data.reset()

predicoes = modelo.predict(
    val_data,
    verbose=1
).ravel()

classes_verdadeiras = val_data.classes

if len(np.unique(classes_verdadeiras)) != 2:
    raise ValueError(
        "A validação precisa possuir as classes 0 e 1."
    )

fpr, tpr, thresholds = roc_curve(
    classes_verdadeiras,
    predicoes
)

sensibilidade_desejada = 0.90

indices_validos = np.where(
    tpr >= sensibilidade_desejada
)[0]

if len(indices_validos) > 0:
    melhor_indice = indices_validos[
        np.argmin(fpr[indices_validos])
    ]

    limiar_fase1 = thresholds[melhor_indice]
    sensibilidade_obtida = tpr[melhor_indice]
    especificidade_obtida = 1 - fpr[melhor_indice]

else:
    limiar_fase1 = 0.5
    sensibilidade_obtida = None
    especificidade_obtida = None

print(f"\nLimiar escolhido: {limiar_fase1:.4f}")

if sensibilidade_obtida is not None:
    print(
        f"Sensibilidade estimada: "
        f"{sensibilidade_obtida:.4f}"
    )

    print(
        f"Especificidade estimada: "
        f"{especificidade_obtida:.4f}"
    )
else:
    print(
        "Nenhum limiar atingiu a sensibilidade desejada. "
        "Foi utilizado o limiar padrão de 0.5."
    )

classes_preditas = (
    predicoes >= limiar_fase1
).astype(int)


# ============================================================
# 3. MATRIZ DE CONFUSÃO
# ============================================================

matriz = confusion_matrix(
    classes_verdadeiras,
    classes_preditas,
    labels=[0, 1]
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "0 (Saudável)",
        "1 (Doente)"
    ],
    yticklabels=[
        "0 (Saudável)",
        "1 (Doente)"
    ]
)

plt.title(
    f"Matriz de Confusão — Fase 1\n"
    f"Limiar = {limiar_fase1:.4f}"
)

plt.xlabel("Classe predita")
plt.ylabel("Classe verdadeira")
plt.tight_layout()
plt.show()


# ============================================================
# 4. RELATÓRIO
# ============================================================

print("\nRelatório de classificação da Fase 1:")

print(
    classification_report(
        classes_verdadeiras,
        classes_preditas,
        target_names=[
            "Saudável",
            "Doente"
        ],
        zero_division=0
    )
)

## Fine-Tuning (Descongelando a Rede)

In [ ]:
print("Recuperando os melhores pesos da Fase 1...")

modelo.load_weights(
    "melhor_modelo_retinopatia.weights.h5"
)

print("Descongelando a rede e mantendo BatchNormalization congelada...")


# Descongela o modelo base
modelo_base.trainable = True


# Mantém as camadas BatchNormalization congeladas
for layer in modelo_base.layers:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False


# Recompila obrigatoriamente depois de alterar trainable
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=criar_metricas()
)


checkpoint_fase2 = ModelCheckpoint(
    "melhor_modelo_retinopatia_fase2.weights.h5",
    monitor="val_pr_auc",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)


early_stop_fase2 = EarlyStopping(
    monitor="val_pr_auc",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1
)


reduce_lr_fase2 = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)


print("Iniciando o treinamento da Fase 2...")


historico_fase2 = modelo.fit(
    treino_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[
        checkpoint_fase2,
        early_stop_fase2,
        reduce_lr_fase2
    ],
    class_weight=pesos_dicionario
)

print("Carregando o melhor checkpoint da Fase 2 para avaliação...")

modelo.load_weights(
    "melhor_modelo_retinopatia_fase2.weights.h5"
)

## Modelo completo


In [ ]:
# Salva o modelo completo: arquitetura + pesos
modelo.save("modelo_retinopatia_final.keras")

print("Modelo completo salvo com sucesso!")

from google.colab import files

files.download("modelo_retinopatia_final.keras")

## Avaliação, Gráficos e Matriz de Confusão (FASE 2)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Gráficos de Acurácia e Erro (Loss) da Fase 2
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(historico_fase2.history['accuracy'], label='Treino', linewidth=2)
plt.plot(historico_fase2.history['val_accuracy'], label='Validação', linewidth=2)
plt.title('Evolução da Acurácia (Binário - Fase 2)')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(historico_fase2.history['loss'], label='Treino', linewidth=2)
plt.plot(historico_fase2.history['val_loss'], label='Validação', linewidth=2)
plt.title('Evolução do Erro (Loss - Binário - Fase 2)')
plt.xlabel('Épocas')
plt.ylabel('Erro')
plt.legend()

plt.show()

# 2. Matriz de Confusão da Fase 2
print("\nA gerar predições para a Matriz de Confusão Binária...")

# Reseta o gerador de validação para garantir a ordem exata das imagens
val_data.reset()

# Probabilidades produzidas pelo modelo após o fine-tuning
predicoes = modelo.predict(
    val_data,
    verbose=1
).ravel()

classes_verdadeiras = val_data.classes

if len(np.unique(classes_verdadeiras)) != 2:
    raise ValueError(
        "A validação precisa possuir exemplos das classes 0 e 1."
    )

# Avalia vários possíveis limiares
fpr, tpr, thresholds = roc_curve(
    classes_verdadeiras,
    predicoes
)

sensibilidade_desejada = 0.90

indices_validos = np.where(
    tpr >= sensibilidade_desejada
)[0]

if len(indices_validos) > 0:
    melhor_indice = indices_validos[
        np.argmin(fpr[indices_validos])
    ]

    limiar_fase2 = thresholds[melhor_indice]
    sensibilidade_obtida = tpr[melhor_indice]
    especificidade_obtida = 1 - fpr[melhor_indice]

else:
    limiar_fase2 = 0.5
    sensibilidade_obtida = None
    especificidade_obtida = None

print(f"\nLimiar escolhido na Fase 2: {limiar_fase2:.4f}")

if sensibilidade_obtida is not None:
    print(
        f"Sensibilidade nesse limiar: "
        f"{sensibilidade_obtida:.4f}"
    )

    print(
        f"Especificidade nesse limiar: "
        f"{especificidade_obtida:.4f}"
    )

classes_preditas = (
    predicoes >= limiar_fase2
).astype(int)

# Desenha a matriz (ATENÇÃO AOS LABELS CORRIGIDOS PARA 2 CLASSES)
matriz = confusion_matrix(classes_verdadeiras, classes_preditas)
plt.figure(figsize=(8, 6))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=['0 (Saudável)', '1 (Doente)'],
            yticklabels=['0 (Saudável)', '1 (Doente)'])
plt.title(
    f"Matriz de Confusão — Fase 2\n"
    f"Limiar = {limiar_fase2:.4f}"
)
plt.xlabel('Classe Predita (IA)')
plt.ylabel('Classe Verdadeira (Médico)')
plt.show()

# 3. Relatório de Métricas Detalhado
print("\nRelatório de classificação na validação — Fase 2:")
print(
    classification_report(
        classes_verdadeiras,
        classes_preditas,
        target_names=[
            "Saudável",
            "Doente"
        ],
        zero_division=0
    )
)